In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
import plotly.graph_objects as go
from scipy.spatial import KDTree
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("No GPU found, using CPU.")
    device = "cpu"

No GPU found, using CPU.


In [6]:
df = pd.read_csv("../data/processed/cleaned_for_embeddings.csv")
print(df.shape)
print(df.columns)
df.head()

(9910, 13)
Index(['names', 'date_x', 'score', 'genre', 'overview', 'crew', 'orig_title',
       'status', 'orig_lang', 'budget_x', 'revenue', 'country', 'content'],
      dtype='str')


,names,date_x,score,genre,overview,crew,orig_title,status,orig_lang,budget_x,revenue,country,content
0,#Alive,06/24/2020,73.0,"Horror, Action, Adventure, Thriller","As a grisly virus rampages a city, a lone man ...","Yoo Ah-in, Oh Joon-woo, Park Shin-hye, Kim Yoo...",#살아있다,Released,Korean,6300000.0,13416285.0,KR,"Title: #Alive. Released in 06/24/2020 , this K..."
1,#FBF,03/31/2022,53.0,Drama,Teenage Annie inadvertently takes her mother's...,"Cree Cicchino, Annie, Gavin Warren, Christian,...",#FBF,Released,English,119600000.0,362461687.2,CA,"Title: #FBF. Released in 03/31/2022 , this CA ..."
2,'71,11/07/2014,68.0,"Thriller, Action, Drama, War",A young British soldier must find his way back...,"Jack O'Connell, Gary Hook, Sean Harris, Captai...",'71,Released,English,11000000.0,3200000.0,AU,"Title: '71. Released in 11/07/2014 , this AU f..."
3,(500) Days of Summer,09/17/2009,73.0,"Comedy, Drama, Romance","Tom, greeting-card writer and hopeless romanti...","Joseph Gordon-Levitt, Tom Hansen, Zooey Descha...",(500) Days of Summer,Released,English,7500000.0,34515303.0,AU,Title: (500) Days of Summer. Released in 09/17...
4,*batteries not included,03/31/1988,67.0,"Science Fiction, Comedy, Family, Fantasy",In a soon to be demolished block of apartments...,"Hume Cronyn, Frank Riley, Jessica Tandy, Faye ...",*batteries not included,Released,English,25000000.0,65088797.0,AU,Title: *batteries not included. Released in 03...


In [7]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
embeddings = model.encode(
    df["content"].tolist(), 
    device=device,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/310 [00:00<?, ?it/s]

In [9]:
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=30,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
coords = reducer.fit_transform(embeddings)

c:\Users\x1Ras\Desktop\Coding\movie-embeddings-visualizer\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [10]:
coords.shape

(9910, 3)

In [11]:
# Map genres to integers
genre_to_num = {g: i for i, g in enumerate(df["genre"].unique())}
color_nums = df["genre"].map(genre_to_num)

# Create 3D scatter using Scatter3d for better control
fig = go.Figure(
    go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode='markers',  # ONLY markers
        marker=dict(
            size=2,        # small markers
            opacity=0.8,
            color=color_nums,  # color by genre
        ),
        text=df["names"],  # hover text
    )
)

# Layout settings
fig.update_layout(
    width=1200,
    height=800,
    scene=dict(
        aspectmode='data'
    ),
    template="plotly_dark",
)

fig.show()

In [13]:
df[["x", "y", "z"]] = coords
coords

array([[ 6.013999 ,  7.710187 ,  9.370611 ],
       [ 2.7877007, 12.405034 ,  7.1003103],
       [ 2.6509306, 11.965941 , 10.011094 ],
       ...,
       [ 1.598295 , 11.255792 ,  9.583212 ],
       [ 4.7252574,  8.792603 ,  9.177561 ],
       [ 3.0251076,  6.6421657,  8.343413 ]],
      shape=(9910, 3), dtype=float32)

In [14]:
tree = KDTree(coords)

In [15]:
def get_closest_movies(movie_name, df, tree, n=5):
    if movie_name not in df['names'].values:
        return "Movie not found, check your spelling."

    idx = df[df['names'] == movie_name].index[0]
    target_xyz = df.loc[idx, ['x', 'y', 'z']].values

    distances, indices = tree.query(target_xyz, k=n + 1)

    # skipping the first one, which is the reference movie
    neighbor_indices = indices[1:]

    return df.iloc[neighbor_indices]

In [17]:
get_closest_movies("Tenet", df, tree).head()

,names,date_x,score,genre,overview,crew,orig_title,status,orig_lang,budget_x,revenue,country,content,x,y,z
9155,"Tomorrow, When the War Began",08/08/2010,62.0,"Action, Adventure, Drama","Ellie Linton, a teen from an Australian coasta...","Caitlin Stasey, Ellie Linton, Rachel Hurd-Wood...","Tomorrow, When the War Began",Released,English,27000000.0,16504936.0,AU,"Title: Tomorrow, When the War Began. Released ...",1.486321,11.722392,9.810475
8998,The X Files: I Want to Believe,07/24/2008,57.0,"Drama, Mystery, Science Fiction, Thriller",Six years after the events of The X-Files seri...,"David Duchovny, Fox Mulder, Gillian Anderson, ...",The X Files: I Want to Believe,Released,English,30000000.0,69363381.0,AU,Title: The X Files: I Want to Believe. Release...,1.352939,11.767612,9.788051
6972,Stealth,09/08/2005,53.0,"Science Fiction, Action, War",Deeply ensconced in a top-secret military prog...,"Josh Lucas, Ben Gannon, Jessica Biel, Kara Wad...",Stealth,Released,English,138000000.0,76416746.0,AU,"Title: Stealth. Released in 09/08/2005 , this ...",1.620307,11.919796,9.855230
8997,The X Files,07/23/1998,68.0,"Mystery, Science Fiction, Thriller","Mulder and Scully, now taken off the FBI's X F...","David Duchovny, Agent Fox Mulder, Gillian Ande...",The X Files,Released,English,66000000.0,189198313.0,AU,"Title: The X Files. Released in 07/23/1998 , t...",1.444950,11.622156,9.864688
2413,Edge of Tomorrow,06/05/2014,76.0,"Action, Science Fiction",Major Bill Cage is an officer who has never se...,"Tom Cruise, Maj. William 'Bill' Cage, Emily Bl...",Edge of Tomorrow,Released,English,178000000.0,367028980.0,AU,Title: Edge of Tomorrow. Released in 06/05/201...,1.482434,11.666023,9.950315


In [17]:
#df.to_csv("../data/processed/with_embeddings.csv") # Do NOT RUN before checking the path, it may overwrite/add to the existing file.

# Experiments
Run the experiments below to support the results section.

In [18]:
def get_color_nums(df):
    genre_to_num = {g: i for i, g in enumerate(df["genre"].unique())}
    return df["genre"].map(genre_to_num)

def plot_3d(coords, df, title):
    color_nums = get_color_nums(df)
    fig = go.Figure(
        go.Scatter3d(
            x=coords[:, 0],
            y=coords[:, 1],
            z=coords[:, 2],
            mode="markers",
            marker=dict(size=2, opacity=0.8, color=color_nums),
            text=df["names"],
        )
    )
    fig.update_layout(
        width=1200,
        height=800,
        scene=dict(aspectmode="data"),
        template="plotly_dark",
        title=title,
    )
    fig.show()

def ensure_bge_model():
    global model
    if "model" not in globals():
        model = SentenceTransformer("BAAI/bge-small-en-v1.5")
    return model

def umap_3d(emb, n_neighbors=30, min_dist=0.1, metric="cosine"):
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric=metric,
    )
    return reducer.fit_transform(emb)

In [19]:
# 1) Dimensionality Reduction Showdown: PCA vs t-SNE vs UMAP
bge_model = ensure_bge_model()
if "embeddings" not in globals():
    embeddings = bge_model.encode(
        df["content"].tolist(),
        device=device,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

pca_coords = PCA(n_components=3, random_state=42).fit_transform(embeddings)
tsne_coords = TSNE(
    n_components=3,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=42,
).fit_transform(embeddings)
umap_coords = umap_3d(embeddings, n_neighbors=30, min_dist=0.1, metric="cosine")

plot_3d(pca_coords, df, "PCA (3D)")
plot_3d(tsne_coords, df, "t-SNE (3D)")
plot_3d(umap_coords, df, "UMAP (3D)")

In [20]:
# 2) Model Baseline Comparison: TF-IDF vs MiniLM vs BGE
query_movie = "The Matrix"
text_series = df["content"].fillna("").astype(str)

def top_knn_dense(emb, df, movie_name, k=5):
    if movie_name not in df["names"].values:
        return None
    idx = df.index[df["names"] == movie_name][0]
    tree = KDTree(emb)
    _, indices = tree.query(emb[idx], k=k + 1)
    neighbors = indices[1:]
    return df.iloc[neighbors]["names"].reset_index(drop=True)

def top_tfidf(tfidf_matrix, df, movie_name, k=5):
    if movie_name not in df["names"].values:
        return None
    idx = df.index[df["names"] == movie_name][0]
    sims = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    sims[idx] = -1
    top_idx = sims.argsort()[::-1][:k]
    return df.iloc[top_idx]["names"].reset_index(drop=True)

tfidf_vectorizer = TfidfVectorizer(stop_words="english", max_features=50000)
tfidf_matrix = tfidf_vectorizer.fit_transform(text_series)
tfidf_top = top_tfidf(tfidf_matrix, df, query_movie, k=5)

minilm_model = SentenceTransformer("all-MiniLM-L6-v2")
minilm_embeddings = minilm_model.encode(
    text_series.tolist(),
    device=device,
    show_progress_bar=True,
    normalize_embeddings=True,
)
minilm_top = top_knn_dense(minilm_embeddings, df, query_movie, k=5)

bge_top = top_knn_dense(embeddings, df, query_movie, k=5)

results_table = pd.DataFrame({
    "TF-IDF": tfidf_top,
    "MiniLM": minilm_top,
    "BGE": bge_top,
})
results_table

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\x1Ras\Desktop\Coding\movie-embeddings-visualizer\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\x1Ras\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/310 [00:00<?, ?it/s]

,TF-IDF,MiniLM,BGE
0,The Matrix Reloaded,The Matrix Revolutions,The Matrix Reloaded
1,The Matrix Revolutions,The Matrix Reloaded,The Matrix Revolutions
2,The Matrix Resurrections,The Matrix Resurrections,The Matrix Resurrections
3,Commando,The Animatrix,Commando
4,The Animatrix,Hackers,The Time Machine


In [21]:
# 3) Mega-Paragraph vs Plot-Only Test
plot_texts = df["overview"].fillna("").astype(str).tolist()
content_texts = df["content"].fillna("").astype(str).tolist()

overview_embeddings = bge_model.encode(
    plot_texts,
    device=device,
    show_progress_bar=True,
    normalize_embeddings=True,
)
content_embeddings = embeddings

overview_coords = umap_3d(overview_embeddings, n_neighbors=30, min_dist=0.1, metric="cosine")
content_coords = umap_3d(content_embeddings, n_neighbors=30, min_dist=0.1, metric="cosine")

plot_3d(overview_coords, df, "UMAP: Plot Only (Overview)")
plot_3d(content_coords, df, "UMAP: Mega-Paragraph (Content)")

Batches:   0%|          | 0/310 [00:00<?, ?it/s]

In [22]:
# 4) UMAP Hyperparameter Tuning: n_neighbors sweep
for n in [5, 30, 100]:
    coords_n = umap_3d(embeddings, n_neighbors=n, min_dist=0.1, metric="cosine")
    plot_3d(coords_n, df, f"UMAP (n_neighbors={n})")